### Scraped Data cleaning

In [1]:
cd /home/ina/Desktop/thesis/Thesis-Project/article-scraping/output

/home/ina/Desktop/thesis/Thesis-Project/article-scraping/output


In [2]:
import pandas as pd
import datetime

## 1. MeinBezirk

In [4]:
bezirk = pd.read_csv("bezirk_flood_articles.csv")

In [113]:
bezirk.head()

,title,url,date,text
0,Bezirk Baden: Große Hilfsbereitschaft für Vere...,https://www.meinbezirk.at/baden/c-lokales/gros...,1. September 2025,"BEZIRK BADEN. Ganze Straßen, Keller, Gebäude u..."
1,"Bezirk Baden: Hochwasser an der Schwechat, Pie...",https://www.meinbezirk.at/baden/c-lokales/hoch...,16. September 2024,BEZIRK BADEN. Von 69 Feuerwehren im Bezirk war...
2,Hochwasser Bilanz 2024: Bilanz nach Überflutun...,https://www.meinbezirk.at/bruck-an-der-leitha/...,23. September 2024,BRUCK/BEZIRK. Entgegen vieler Hoffnung hörte d...
3,EU-Solidaritätsfonds: 42 Millionen Euro für Ös...,https://www.meinbezirk.at/c-lokales/42-million...,8. Dezember 2025,Österreich erhält neben einer bereits im April...
4,Überschwemmungen und Sperren: Aktuelle Hochwas...,https://www.meinbezirk.at/enns/c-lokales/aktue...,zuletzt aktualisiert am 16. September 2024,"""Trotz des immer noch andauernden Dauerregens ..."


### 1. Deal with date variable

In [5]:
# singular line is wonky

bezirk.loc[4, "date"] = "16. September 2024"

In [6]:

for i, d in enumerate(bezirk["date"]):
    
    if pd.isna(d):
        continue
    else:
        
        date_str = d

        # Dictionary für die Übersetzung
        months = {
            "Januar": "January", "Februar": "February", "März": "March",
            "April": "April", "Mai": "May", "Juni": "June",
            "Juli": "July", "August": "August", "September": "September",
            "Oktober": "October", "November": "November", "Dezember": "December"
        }

        # Monat ersetzen
        for de, en in months.items():
            if de in date_str:
                date_str = date_str.replace(de, en)
                break

        # Jetzt funktioniert strptime mit %B (ausgeschriebener Name)
        date_obj = datetime.datetime.strptime(date_str, "%d. %B %Y")
        bezirk.loc[i, "date"] = date_obj
        #bezirk_trial["date"].replace(d, date_obj)


## 2. ORF

In [7]:
orf = pd.read_csv("orf_flood_articles.csv")

In [8]:
orf.head()

,title,metadata,abstract,figcaption,body
0,Hochwasserschutz in Feldkirch auf der Zielgeraden,"Verkehr (Vorarlberg) 05.04.2026, 15:26",Das Projekt „Hochwasserschutz Feldkirch“ geht ...,NaN,Seit 2022 wird am Hochwasserschutz-Projekt in ...
1,Wie das March-Hochwasser Dürnkrut prägte,"Chronik (Österreich) 03.04.2026, 19:19",Starke Regenfälle haben vor 20 Jahren zu einem...,Am 3. April 2006 um 3.30 Uhr gab es in Dürnkru...,Vor genau 20 Jahren ereignete sich eines der s...
2,Beschwerde gegen Hochwasserschutzprojekt,"Chronik (Salzburg) 28.03.2026, 08:44",Die Landesumweltanwaltschaft (LUA) hat gegen d...,NaN,Der Standort liege laut LUA in mehreren besond...
3,Hochwasserschutz: Bürgermeister nehmen Stellung,"Wissenschaft (Steiermark) 18.03.2026, 07:31",Der Landesrechnungshof hat am Montag zum Thema...,NaN,"Der Rechnungshof ist der Meinung, dass die Gem..."
4,Hochwasserschutz: Noch immer zu lange Dauer,"Politik Inland (Steiermark) 17.03.2026, 12:34",Der Landesrechnungshof hat nach 2018 wieder de...,NaN,Im letzten Jahr blieb die Steiermark von schli...


In [9]:
# dealing with the date column

dates_str = []

# extract the date from the metadata column
for line in orf["metadata"].str.split(" "):
    if len(line) == 3:
        dates_str.append(line[1])
    if len(line) == 4:
        dates_str.append(line[2])
    if len(line) == 5:
        dates_str.append(line[3])

# remove the "," that's still in the string
for j, i in enumerate(dates_str):
    dates_str[j]= i.split(",")[0]
    
date_orf = []

# convert to datetimie format
for i in dates_str:
    x = datetime.datetime.strptime(i, "%d.%m.%Y")
    date_orf.append(x.strftime("%Y-%m-%d %H:%M:%S"))
    
# add the date column to the orf dateframe
orf["date"] = pd.Series(date_orf)

In [10]:
# combine all text columns into one column (is this too messy?)

orf["all_text"] = [None] * len(orf)

for i in range(len(orf)):
    
    abstract = orf["abstract"].fillna("")
    body = orf["body"].fillna("")
    caption = orf["figcaption"].fillna("")
    
    orf["all_text"].iloc[i] = abstract.loc[i] + " | " + body.loc[i] + " | " + caption.loc[i]


/tmp/ipykernel_5427/3926690422.py:11: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  orf["all_text"].iloc[i] = abstract.loc[i] + " | " + body.loc[i] + " | " + caption.loc[i]
/tmp/ipykernel_5427/3926690422.py:11: FutureWarning: ChainedAssignme

## 3. Heute

In [11]:
heute = pd.read_csv("heute_flood_articles.csv")

In [56]:
heute.head()

,title,sub_header,body,date,url
0,1 Mio. Euro Schaden: Unternehmer wartet auf Fl...,Vor 3 Monaten zerstörte das Hochwasser dieses ...,"""Obwohl die Kommission längst bei mir war und ...","09.12.2024, 21:16",https://www.heute.at/s/1-mio-euro-schaden-unte...
1,10 Tage vor Wahl platzt Elefantenrunde auf Ser...,Es sollte der Startschuss in die heiße Wahlkam...,Die Folgen derJahrhundertflutwirbeln auch den ...,"18.09.2024, 18:47",https://www.heute.at/s/10-tage-vor-wahl-platzt...
2,10 Verletzte durch Flut in Wien – jetzt alle P...,"Tief ""Anett"" hält ganz Österreich weiterhin in...",Die Hochwassersituation in Niederösterreich un...,"16.09.2024, 15:54",https://www.heute.at/s/10-verletzte-durch-flut...
3,100.000 Rinder im Todeskampf – Sintflut in Aus...,Eine Fläche in der Größe Frankreichs ist in Au...,"""Down Under"" steht unter Schock: In dem von Vi...","01.04.2025, 17:23",https://www.heute.at/s/100000-rinder-im-todesk...
4,114 Tote nach Kalmaegi – nächster Taifun näher...,Taifun Kalmaegi hat auf den Philippinen 114 Me...,Auf den Philippinen ist die Zahl der Toten dur...,"06.11.2025, 09:06",https://www.heute.at/s/114-tote-nach-kalmaegi-...


In [12]:
datetime.datetime.strptime(heute["date"][0], "%d.%m.%Y, %H:%M")

datetime.datetime(2024, 12, 9, 21, 16)

In [13]:
for i in range(len(heute["date"])):
    x = datetime.datetime.strptime(heute["date"][i], "%d.%m.%Y, %H:%M")
    heute["date"][i] = x.strftime("%Y-%m-%d %H:%M:%S")


/tmp/ipykernel_5427/2961826804.py:3: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  heute["date"][i] = x.strftime("%Y-%m-%d %H:%M:%S")


### Concatenate all dataframes

In [14]:
# make sure orf df fits the format

df_orf = orf[["title", "all_text", "date"]]
df_orf["source"] = ["ORF"] * len(df_orf)
df_orf["url"] = [None] * len(df_orf)
df_orf.rename(columns={"all_text": "body"}, inplace = True)

/tmp/ipykernel_5427/2556828251.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_orf["source"] = ["ORF"] * len(df_orf)
/tmp/ipykernel_5427/2556828251.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_orf["url"] = [None] * len(df_orf)
/tmp/ipykernel_5427/2556828251.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [15]:
# make sure heute df fits the format
heute["source"] = ["HEUTE"] * len(heute)
df_heute = heute[["title", "body", "date", "source", "url"]]

In [16]:
# make sure meinbezirk df fits the format
bezirk["source"] = ["meinbezirk"] * len(bezirk)
df_bezirk = bezirk[["title", "text", "date", "source", "url"]]
df_bezirk.rename(columns={"text": "body"}, inplace = True)

In [17]:
df = pd.concat([df_orf, df_heute, df_bezirk], ignore_index = True)

In [18]:
len(df)

1181

## excluding articles that obviously don't fit

Zwischen 09/2024 und 12/2024 eingrenzen

In [19]:
import datetime
# Keep this as a datetime object
min_date = datetime.datetime.strptime("2024-09-01 00:00:00", "%Y-%m-%d %H:%M:%S")
max_date = datetime.datetime.strptime("2024-12-31 00:00:00", "%Y-%m-%d %H:%M:%S")

# Ensure your column is also datetime
df["date"] = pd.to_datetime(df["date"])

# Now the comparison works perfectly
df = df[(df["date"] >= min_date) & (df["date"] <= max_date)]

In [ ]:
# inidices to be removed:
# 214 (Hochwasserschutzprojekt Götzis)
# 108 Salzburg (Artikel dazu, dass nur wenige Hochwasserschutzprojekte in Salzburg)
# 127 Hochwasserschutz für Tauerntäler wird abgespeckt	
# 149: Hochwasserschutzprojekt in Unken fertiggestellt
# 125: Pilotprojekt für Hochwasserschutz in Lafnitz
# 173 Igel in großer Not nach Hochwasser
# 216 „Trockene Hochwasser“ werden häufiger
# 235 Hochwasser: Salzburg glimpflich davongekommen
# 289 1,5 Millionen Blitze über Österreich registriert
# 243 Villach baut Hochwasserschutz
# 239 Diesel flutet Autobahn – Feuerwehr errichtet S...
# 403	Dickster Python schlängelt sich durch das Rege...	
# 363 Klimakrise mischte auch im Oktober kräftig mit
# 438	Es ist offiziell Sommer! – Wärmewucht trifft Ö...
# 248	Unwetter in Polen: Wroclaw ruft Hochwasseralar..
# 239	Orban verschiebt wegen Hochwassers Rede vor EU...
# 226	Polen: Polizei und Armee gegen Plünderer in Ho...
# 234	Mehrere Tote in Polen durch Hochwasser
# 743	Nehammer reist jetzt zu Hochwasser-Gipfel nach...
# 204	Trotz Hochwassers: Auftakt zu Regionalwahl in ..
# 232	Hochwasserlage auch in Tschechien weiter kritisch
# 247	Hochwasser in Tschechien: Toter und viele Verm...
# 1080	Zwei Hochwasserschutz-Projekte in NÖ fertigges...

# filtered_df = df[["Hochwasserschutz-Projekt" in i for i in df["title"]]]


In [ ]:
rows_to_drop = [214, 108, 127, 149, 125, 173, 216, 235, 289, 243, 239, 403, 363, 438, 248, 239, 226, 234, 743, 204, 232, 247, 1080]
df.drop(rows_to_drop,axis=0, inplace= True)

In [102]:
full_df.head()

,title,body,date,source,url
103,Hochwasser: ÖBB-Züge öfter verspätet,Mehr als 600.000 Züge sind heuer laut ÖBB im W...,2024-12-31,ORF,None
104,Hochwasser 2024: „Die Angst bleibt“,Ein Jahrhunderthochwasser hat Niederösterreich...,2024-12-29,ORF,None
105,"Hochwasser, Bombendrohungen und große Emotionen",Das Jahr 2024 neigt sich dem Ende zu. Und wir ...,2024-12-28,ORF,None
106,Hochwasserschutz wird ausgebaut,Weil Starkwetterereignisse wie das tausendjähr...,2024-12-28,ORF,None
107,Hochwasser im Juni hinterließ Spur der Verwüstung,Die heftigen Unwetter Anfang Juni im Südburgen...,2024-12-28,ORF,None


In [ ]:
# # save as csv

# full_df.to_csv("full_df.csv", index = False)

In [95]:
df.iloc[4].body

'Die heftigen Unwetter Anfang Juni im Südburgenland haben das Leben vieler Burgenländerinnen und Burgenländer plötzlich auf den Kopf gestellt. In einer Nacht hat es im Südburgenland so viel geregnet wie normalerweise in einem Monat. | Es war die Nacht von 8. auf 9. Juni, als Unwetter einen kompletten Ausnahmezustand im Bezirk Oberwart auslösten. Kleine Bäche wurden zu reißenden Flüssen, Straßen wurden überflutet und zahlreiche Häuser sind unter Wasser gestanden – mehr dazu in Enorme Schäden durch Hochwasser.\n„Einige Familien haben alles verloren“, sagte Ralf Knauseder aus Schreibersdorf einen Tag nach dem historischen Hochwasser. Was sie in dieser Nacht erlebt hätten, sei „unpackbar“ gewesen, so der Schreibersdorfer. „Es ist tragisch für alle“, sagte Karina Böcskör aus Oberwart, auch ihr Keller stand unter Wasser.\nTausende Feuerwehrmitglieder und das Bundesheer waren tagelang im Dauereinsatz – mehr dazu in Hochwasser: Lage weiterhin angespannt. Das Hochwasser forderte auch ein Todeso

In [98]:
full_df.iloc[0:2]

,title,body,date,source,url
103,Hochwasser: ÖBB-Züge öfter verspätet,Mehr als 600.000 Züge sind heuer laut ÖBB im W...,2024-12-31,ORF,None
104,Hochwasser 2024: „Die Angst bleibt“,Ein Jahrhunderthochwasser hat Niederösterreich...,2024-12-29,ORF,None


#### Checking out the Articles after the LLM categorization

In [ ]:
articles_df = pd.read_csv("/home/ina/Desktop/thesis/Thesis-Project/article-scraping/output/articles_df.csv")
filtered_df = articles_df[["Eberndorf" in i for i in articles_df["title"]]]
filtered_df